## Alzheimer Detection using Google Colaboratory

### Step 0: Import Libraries and Clone Repository

In [ ]:
%cd /content/
!git clone https://github.com/Verbosi7y/ai-alzheimer-detection.git

In [ ]:
%pip install --upgrade pip
%pip install torch
%pip install numpy
%pip install matplotlib
%pip install seaborn
%pip install scikit-image
%pip install scikit-learn
%pip install imbalanced-learn
%pip install albumentations
%pip install opencv-python
%pip install pillow

In [ ]:
# Uncomment if you are running Google Colab on a CUDA GPU (NVIDIA)
# %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [ ]:
from torch.utils.data import DataLoader

import os
import sys

Setting Paths

In [ ]:
parent_path = r"/content/ai-alzheimer-detection"

kaggle_dir = r"assets/Kaggle"
kaggle_path = os.path.join(parent_path, kaggle_dir)

kaggle_dataset_dir = r"alzheimer_mri_preprocessed_dataset"
kaggle_raw_dir = r"alzheimer_mri_preprocessed_dataset/raw"

kaggle_dataset_path = os.path.join(kaggle_path, kaggle_dataset_dir)
kaggle_raw_path = os.path.join(kaggle_path, kaggle_raw_dir)

model_dir = r"models"
model_path = os.path.join(parent_path, model_dir)

if not os.path.exists(model_path):
    os.makedirs(model_path)

model_dir = r"models/best_ad_model.pth"
model_path = os.path.join(parent_path, model_dir)

In [ ]:
# add parent to path
sys.path.append(parent_path)

### Step 1: Load the Dataset

In [ ]:
from alzheimersdetection import Dataset

X, y = Dataset.step1_load_data(path=kaggle_raw_path) # np.array, np.array

### Step 2: Split the dataset

Split the data into 80% training and 20% testing data. Ensure same class distribution using stratify=y (class/label).

Further split the training data into 75% training and 25% validation respectively.

Ratio: 60% Training : 20% Validation : 20% Testing

In [ ]:
test_size = 0.20
validation_size = 0.25

split_dataset = Dataset.step2_split_data(X, y, test_size=test_size, validation_size=validation_size)

Visualization for the Distribution of the Training Dataset

Results should be heavily imbalanced 

In [ ]:
import stats.statistics as Statistics

title_before_aug = "AD Classification Distribution of Training Dataset"

sample_dist = Dataset.distribution(split_dataset["train"]["y"])

Statistics.pieChartClassificationPlot(sample_dist, title_before_aug)

### Step 3: Balance and Oversample the Dataset

#### 3a. Balance

To further balance the dataset, we need to employ more techniques. One of which is data augmentation.
Method to balance the data augmentation process is to define class-specific augmentation rates.

In [ ]:
'''
    Rates:
    - Non_Demented: 1
    - Very_Mild_Demented: 1
    - Mild_Demented: 2
    - Moderate_Demented: 5
'''
rates = [1, 1, 2, 5]

split_dataset["train"] = Dataset.step3a_augmentation(split_dataset["train"], rates=rates)

Dataset.display_split(split_dataset=split_dataset)

Visualizing out results of the class distribution after data augmentation

In [ ]:
title_after_aug = "AD Classification Distribution after Data Augmentation"

aug_dist = Dataset.distribution(split_dataset["train"]["y"])

Statistics.pieChartClassificationPlot(aug_dist, title_after_aug)

#### 3b. ADASYN Oversampling

The dataset is still imbalanced and to fix this, we need to increase the minority class's representation (oversampling). This allows us to have a more balanced dataset.

We will be using Adaptive Synthetic Sampling (ADASYN) to oversample the minority classes.

In [ ]:
# Visualize class imbalance before ADASYN
title_before_ADASYN = "Class Distribution before ADASYN"

Dataset.display_split(split_dataset=split_dataset);

Statistics.ad_plot_bar(sample=split_dataset["train"], title=title_before_ADASYN)

Applying Adapative Synthetic Sampling (ADASYN)

Optimal Results: ~25% distribution across all AD classifications.

In [ ]:
k = 5 # This is the k-neighbors which will be used for ADASYN

split_dataset["train"] = Dataset.step3b_ADASYN(sample=split_dataset["train"], k=k)

Visualizing our results after applying ADASYN as a Bar Plot

In [ ]:
# Visualize class imbalance after ADASYN
title_after_ADASYN = "Class Distribution after ADASYN"

Dataset.display_split(split_dataset=split_dataset);

Statistics.ad_plot_bar(sample=split_dataset["train"], title=title_after_ADASYN)

### Step 4: Save as dataset .npz and images

In [ ]:
Dataset.step4_save_npz(split_dataset, path=kaggle_dataset_path)

### Step 5: Define Hyperparameters

In [ ]:
param = {
        "epoches"       : 25, # implement early stopping
        "learning_rate" : 0.001,
        "batch_size"    : 8,
        "early_stop"    : 5
        }

### Step 6: Load Dataset as Dataloader

In [ ]:
from alzheimersdetection.AlzheimerDataset import AlzheimerDataset

train_dataset = AlzheimerDataset(samples=split_dataset["train"])
val_dataset = AlzheimerDataset(samples=split_dataset["test"])
test_dataset = AlzheimerDataset(samples=split_dataset["validation"])

loaders =  {
           "train"  : DataLoader(train_dataset, batch_size=param["batch_size"], shuffle=True),
           "test"   : DataLoader(val_dataset, batch_size=param["batch_size"], shuffle=False),
           "val"    : DataLoader(test_dataset, batch_size=param["batch_size"], shuffle=False)
           }

### Step 7: Device Setup

Device Setup:

If you have an Nvidia GPU, you need to install CUDA

Otherwise, CPU will be used

In [ ]:
from alzheimersdetection import AlzheimerModel

device = AlzheimerModel.set_device()

### Step 8: Creating CNN Model

Create our model using PyTorch's Convolutional Neural Network

In [ ]:
from alzheimersdetection.AlzheimerModel import AlzheimerCNN

model = AlzheimerCNN().to(device)

### Step 9: Train the Model

Criterion: Cross Entropy Loss

Optimizer: Adam

In [ ]:
AlzheimerModel.step9_train_model(model, param, loaders, device, model_path)

### Step 10: Verify using Test Data

In [ ]:
from alzheimersdetection import AlzheimerMetrics

AlzheimerMetrics.run_metrics(model, loaders["test"], device)

# New Code

In [1]:
import os
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2, EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, f1_score

print("TensorFlow version:", tf.__version__)
print("Running on Kaggle environment" if os.path.isdir("/kaggle/input") else "Running locally")

# Paths
IS_KAGGLE = os.path.isdir("/kaggle/input")
if IS_KAGGLE:
    DATASET_DIR = "/kaggle/input/alzheimers-detection-dataset/dataset/"
    MODELS_DIR = "/kaggle/working/models"
else:
    DATASET_DIR = os.path.join(os.getcwd(), "dataset")
    MODELS_DIR = os.path.join(os.getcwd(), "models")

SAVED_MODEL_PATH = os.path.join(MODELS_DIR, "best_alzheimer_model.keras")
CLASS_NAMES_PATH = os.path.join(MODELS_DIR, "class_names.txt")

# Training
CLASS_NAMES = ["Mild_Demented", "Moderate_Demented", "Non_Demented", "Very_Mild_Demented"]
NUM_CLASSES = len(CLASS_NAMES)
IMG_SIZE = (224, 224)
IMG_SHAPE = (*IMG_SIZE, 3)
BATCH_SIZE = 32
SEED = 42
VAL_SPLIT = 0.2
TEST_SPLIT = 0.1
EPOCHS = 50
PATIENCE = 10
OVERSAMPLE_MIN_PER_CLASS = 800
CLASS_WEIGHT_MAX = 10.0
IMG_EXTENSIONS = (".png", ".jpg", ".jpeg", ".bmp", ".gif")

tf.random.set_seed(SEED)
np.random.seed(SEED)

os.makedirs(MODELS_DIR, exist_ok=True)
print("Dataset:", DATASET_DIR)
print("Output:", MODELS_DIR)


def collect_image_paths_and_labels():
    image_paths, labels = [], []
    for class_idx, class_name in enumerate(CLASS_NAMES):
        class_dir = os.path.join(DATASET_DIR, class_name)
        if not os.path.isdir(class_dir):
            continue
        for fname in os.listdir(class_dir):
            if fname.lower().endswith(IMG_EXTENSIONS):
                image_paths.append(os.path.join(class_dir, fname))
                labels.append(class_idx)
    return np.array(image_paths), np.array(labels)


def load_and_split_data():
    paths, labels = collect_image_paths_and_labels()
    if len(paths) == 0:
        raise FileNotFoundError("No images in " + str(DATASET_DIR) + " for " + str(CLASS_NAMES))
    X_trainval, X_test, y_trainval, y_test = train_test_split(
        paths, labels, test_size=TEST_SPLIT, stratify=labels, random_state=SEED
    )
    val_ratio = VAL_SPLIT / (1 - TEST_SPLIT)
    X_train, X_val, y_train, y_val = train_test_split(
        X_trainval, y_trainval, test_size=val_ratio, stratify=y_trainval, random_state=SEED
    )
    return (X_train, y_train), (X_val, y_val), (X_test, y_test)


def oversample_minority_classes(X_train, y_train, target_min=OVERSAMPLE_MIN_PER_CLASS):
    X_list, y_list = [], []
    for c in range(NUM_CLASSES):
        mask = y_train == c
        X_c, y_c = X_train[mask], y_train[mask]
        n = len(X_c)
        if n == 0:
            continue
        n_repeat = max(1, (target_min + n - 1) // n)
        indices = np.tile(np.arange(n), n_repeat)[:max(n, target_min)]
        np.random.RandomState(SEED).shuffle(indices)
        X_list.append(X_c[indices])
        y_list.append(np.full(len(indices), c))
    X_bal = np.concatenate(X_list, axis=0)
    y_bal = np.concatenate(y_list, axis=0)
    perm = np.random.RandomState(SEED).permutation(len(X_bal))
    return X_bal[perm], y_bal[perm]


def get_class_weights(y, max_weight=CLASS_WEIGHT_MAX):
    weights = compute_class_weight("balanced", classes=np.unique(y), y=y)
    return {i: min(float(w), max_weight) for i, w in enumerate(weights)}


# Load and oversample
(X_train, y_train), (X_val, y_val), (X_test, y_test) = load_and_split_data()
print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
X_train, y_train = oversample_minority_classes(X_train, y_train)
print(f"After oversampling - Train: {len(X_train)}")
class_weights = get_class_weights(y_train)
print("Class weights (capped):", class_weights)


def _load_and_preprocess(path, label, augment=False):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    if augment:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_brightness(img, 0.1)
        img = tf.image.random_contrast(img, 0.9, 1.1)
        img = tf.clip_by_value(img, 0, 1)
    return img, label


def dataset_from_paths(paths, labels, batch_size, shuffle=True, augment=False):
    ds = tf.data.Dataset.from_tensor_slices((paths.astype(str), labels.astype(np.int32)))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(paths), 2048), seed=SEED)
    ds = ds.map(
        lambda p, l: _load_and_preprocess(p, l, augment=augment),
        num_parallel_calls=tf.data.AUTOTUNE,
    )
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


train_ds = dataset_from_paths(X_train, y_train, BATCH_SIZE, shuffle=True, augment=True)
val_ds = dataset_from_paths(X_val, y_val, BATCH_SIZE, shuffle=False)
test_ds = dataset_from_paths(X_test, y_test, BATCH_SIZE, shuffle=False)
print("Train/val/test datasets ready.")

def build_custom_cnn():
    inputs = keras.Input(shape=IMG_SHAPE)
    x = layers.Conv2D(64, 3, activation="relu", padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPool2D(2)(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Conv2D(128, 3, activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPool2D(2)(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Conv2D(256, 3, activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="custom_cnn")


def build_mobilenetv2():
    base = MobileNetV2(input_shape=IMG_SHAPE, include_top=False, weights="imagenet")
    base.trainable = False
    inputs = keras.Input(shape=IMG_SHAPE)
    x = base(inputs)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="mobilenetv2")


def build_efficientnetb0():
    base = EfficientNetB0(input_shape=IMG_SHAPE, include_top=False, weights="imagenet")
    base.trainable = False
    inputs = keras.Input(shape=IMG_SHAPE)
    x = base(inputs)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="efficientnetb0")


MODEL_BUILDERS = {
    "custom_cnn": build_custom_cnn,
    "mobilenetv2": build_mobilenetv2,
    "efficientnetb0": build_efficientnetb0,
}

def train_and_evaluate(model_name, train_ds, val_ds, test_ds, y_test, class_weights):
    model = MODEL_BUILDERS[model_name]()
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    ]
    history = model.fit(
        train_ds, validation_data=val_ds, epochs=EPOCHS,
        class_weight=class_weights, callbacks=callbacks, verbose=1,
    )
    y_pred = np.argmax(model.predict(test_ds), axis=1)
    test_acc = np.mean(y_test == y_pred)
    test_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
    report = classification_report(y_test, y_pred, target_names=CLASS_NAMES, zero_division=0)
    cm = confusion_matrix(y_test, y_pred).tolist()
    return {
        "model": model,
        "history": history.history,
        "test_accuracy": float(test_acc),
        "test_f1_macro": float(test_f1),
        "classification_report": report,
        "confusion_matrix": cm,
    }

print("Function train_and_evaluate defined.")

results = {}
for name in MODEL_BUILDERS:
    print(f"\n{'='*60}\nTraining {name.upper()}\n{'='*60}")
    try:
        results[name] = train_and_evaluate(
            name, train_ds, val_ds, test_ds, y_test, class_weights
        )
        print(f"Test accuracy : {results[name]['test_accuracy']:.4f}")
        print(f"Test macro F1 : {results[name]['test_f1_macro']:.4f}")
    except Exception as e:
        print(f"Error: {e}")
        results[name] = None

best_name = None
best_f1 = -1
for name, res in results.items():
    if res and res["test_f1_macro"] > best_f1:
        best_f1 = res["test_f1_macro"]
        best_name = name

if best_name is None:
    raise RuntimeError("No model trained successfully.")

print(f"Best model: {best_name} (test macro F1: {best_f1:.4f})")
best_model = results[best_name]["model"]
best_model.save(SAVED_MODEL_PATH)
with open(CLASS_NAMES_PATH, "w") as f:
    f.write("\n".join(CLASS_NAMES))

metrics_path = os.path.join(MODELS_DIR, "best_model_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(
        {
            "best_model": best_name,
            "test_accuracy": results[best_name]["test_accuracy"],
            "test_f1_macro": results[best_name]["test_f1_macro"],
            "classification_report": results[best_name]["classification_report"],
            "confusion_matrix": results[best_name]["confusion_matrix"],
        },
        f,
        indent=2,
    )

print(f"Model saved → {SAVED_MODEL_PATH}")
print(f"Class names → {CLASS_NAMES_PATH}")
print(f"Metrics     → {metrics_path}")
print("\nClassification report:\n", results[best_name]["classification_report"])





2026-02-20 23:25:48.901374: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771629949.085251      33 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771629949.142849      33 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


TensorFlow version: 2.18.0
Running on Kaggle environment
Dataset: /kaggle/input/alzheimers-detection-dataset/dataset/
Output: /kaggle/working/models
Train: 4480, Val: 1280, Test: 640
After oversampling - Train: 5408
Class weights (capped): {0: 1.69, 1: 1.69, 2: 0.6035714285714285, 3: 0.8622448979591837}


I0000 00:00:1771629961.718991      33 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1771629961.721431      33 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Train/val/test datasets ready.
Function train_and_evaluate defined.

Training CUSTOM_CNN
Epoch 1/50


I0000 00:00:1771629967.974479      97 service.cc:148] XLA service 0x7c7b4c012350 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1771629967.975342      97 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1771629967.975362      97 service.cc:156]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1771629968.553804      97 cuda_dnn.cc:529] Loaded cuDNN version 90300


  1/169 ━━━━━━━━━━━━━━━━━━━━ 41:58 15s/step - accuracy: 0.2188 - loss: 1.2698

I0000 00:00:1771629978.925933      97 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


169/169 ━━━━━━━━━━━━━━━━━━━━ 36s 126ms/step - accuracy: 0.3827 - loss: 1.3269 - val_accuracy: 0.5000 - val_loss: 4.1394 - learning_rate: 0.0010
Epoch 2/50
169/169 ━━━━━━━━━━━━━━━━━━━━ 18s 107ms/step - accuracy: 0.4695 - loss: 1.1822 - val_accuracy: 0.5000 - val_loss: 3.4898 - learning_rate: 0.0010
Epoch 3/50
169/169 ━━━━━━━━━━━━━━━━━━━━ 18s 108ms/step - accuracy: 0.5040 - loss: 1.0673 - val_accuracy: 0.4969 - val_loss: 1.4941 - learning_rate: 0.0010
Epoch 4/50
169/169 ━━━━━━━━━━━━━━━━━━━━ 19s 110ms/step - accuracy: 0.5343 - loss: 0.9780 - val_accuracy: 0.4969 - val_loss: 0.9820 - learning_rate: 0.0010
Epoch 5/50
169/169 ━━━━━━━━━━━━━━━━━━━━ 19s 111ms/step - accuracy: 0.5608 - loss: 0.8684 - val_accuracy: 0.5117 - val_loss: 1.3463 - learning_rate: 0.0010
Epoch 6/50
169/169 ━━━━━━━━━━━━━━━━━━━━ 19s 114ms/step - accuracy: 0.5665 - loss: 0.8252 - val_accuracy: 0.3516 - val_loss: 2.7907 - learning_rate: 0.0010
Epoch 7/50
169/169 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step - accuracy: 0.5685 - loss: 

In [ ]:
!pip install config

In [3]:
# --- Imports ---
import os
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2, EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# --- Config ---
IS_KAGGLE = os.path.isdir("/kaggle/input")
DATASET_DIR = "/kaggle/input/alzheimers-detection-dataset/dataset" if IS_KAGGLE else os.path.join(os.getcwd(), "dataset")
MODELS_DIR = "/kaggle/working/models" if IS_KAGGLE else os.path.join(os.getcwd(), "models")
SAVED_MODEL_PATH = os.path.join(MODELS_DIR, "best_alzheimer_model.keras")
CLASS_NAMES_PATH = os.path.join(MODELS_DIR, "class_names.txt")
CLASS_NAMES = ["Mild_Demented", "Moderate_Demented", "Non_Demented", "Very_Mild_Demented"]
NUM_CLASSES = len(CLASS_NAMES)
IMG_SIZE = (224, 224)
IMG_SHAPE = (*IMG_SIZE, 3)
BATCH_SIZE = 32
SEED = 42
VAL_SPLIT, TEST_SPLIT = 0.2, 0.1
EPOCHS, PATIENCE = 55, 12
OVERSAMPLE_MIN_PER_CLASS, CLASS_WEIGHT_MAX = 1200, 10.0
LABEL_SMOOTHING, FINETUNE_EPOCHS, FINETUNE_LR = 0.1, 15, 1e-5
UNFREEZE_FRACTION = 0.25
IMG_EXTENSIONS = (".png", ".jpg", ".jpeg", ".bmp", ".gif")
tf.random.set_seed(SEED)
np.random.seed(SEED)
os.makedirs(MODELS_DIR, exist_ok=True)
print("TensorFlow:", tf.__version__, "| Kaggle" if IS_KAGGLE else "| Local", "| Dataset:", DATASET_DIR)

# --- Data: collect, split, oversample, class weights ---
def collect_paths_labels():
    paths, labels = [], []
    for i, cname in enumerate(CLASS_NAMES):
        d = os.path.join(DATASET_DIR, cname)
        if not os.path.isdir(d): continue
        for f in os.listdir(d):
            if f.lower().endswith(IMG_EXTENSIONS):
                paths.append(os.path.join(d, f))
                labels.append(i)
    return np.array(paths), np.array(labels)

paths, labels = collect_paths_labels()
if len(paths) == 0: raise FileNotFoundError("No images in " + str(DATASET_DIR))
X_tv, X_test, y_tv, y_test = train_test_split(paths, labels, test_size=TEST_SPLIT, stratify=labels, random_state=SEED)
val_ratio = VAL_SPLIT / (1 - TEST_SPLIT)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=val_ratio, stratify=y_tv, random_state=SEED)

def oversample(X, y, target=OVERSAMPLE_MIN_PER_CLASS):
    Xl, yl = [], []
    for c in range(NUM_CLASSES):
        m = y == c
        Xc, nc = X[m], m.sum()
        if nc == 0: continue
        idx = np.tile(np.arange(nc), (target + nc - 1) // nc)[:max(nc, target)]
        np.random.RandomState(SEED).shuffle(idx)
        Xl.append(Xc[idx]); yl.append(np.full(len(idx), c))
    out_x = np.concatenate(Xl); out_y = np.concatenate(yl)
    p = np.random.RandomState(SEED).permutation(len(out_x))
    return out_x[p], out_y[p]

X_train, y_train = oversample(X_train, y_train)
class_weights = {i: min(float(w), CLASS_WEIGHT_MAX) for i, w in enumerate(compute_class_weight("balanced", classes=np.unique(y_train), y=y_train))}
print("Train:", len(X_train), "Val:", len(X_val), "Test:", len(X_test), "| Class weights:", class_weights)

# --- tf.data with augmentation ---
def load_preprocess(path, label, aug=False):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    if aug:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_flip_up_down(img)
        img = tf.image.random_brightness(img, 0.2)
        img = tf.image.random_contrast(img, 0.85, 1.15)
        img = tf.image.random_saturation(img, 0.9, 1.1)
        k = tf.random.uniform([], 0, 4, dtype=tf.int32)
        img = tf.image.rot90(img, k)
        img = tf.clip_by_value(img, 0, 1)
    return img, label

def make_ds(paths, labels, batch, shuffle=True, augment=False):
    ds = tf.data.Dataset.from_tensor_slices((paths.astype(str), labels.astype(np.int32)))
    if shuffle: ds = ds.shuffle(min(len(paths), 2048), seed=SEED)
    ds = ds.map(lambda p, l: load_preprocess(p, l, augment), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(X_train, y_train, BATCH_SIZE, True, True)
val_ds = make_ds(X_val, y_val, BATCH_SIZE, False)
test_ds = make_ds(X_test, y_test, BATCH_SIZE, False)

# --- Models ---
def build_custom_cnn():
    inp = keras.Input(shape=IMG_SHAPE)
    x = layers.Conv2D(96, 3, activation="relu", padding="same")(inp)
    x = layers.BatchNormalization()(x); x = layers.MaxPool2D(2)(x); x = layers.Dropout(0.2)(x)
    x = layers.Conv2D(192, 3, activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x); x = layers.MaxPool2D(2)(x); x = layers.Dropout(0.3)(x)
    x = layers.Conv2D(384, 3, activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x); x = layers.MaxPool2D(2)(x); x = layers.Dropout(0.3)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.5)(x)
    return keras.Model(inp, layers.Dense(NUM_CLASSES, activation="softmax")(x), name="custom_cnn")

def build_mobilenetv2():
    base = MobileNetV2(input_shape=IMG_SHAPE, include_top=False, weights="imagenet")
    base.trainable = False
    inp = keras.Input(shape=IMG_SHAPE)
    x = base(inp); x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x); x = layers.Dropout(0.4)(x)
    return keras.Model(inp, layers.Dense(NUM_CLASSES, activation="softmax")(x), name="mobilenetv2")

def build_efficientnetb0():
    base = EfficientNetB0(input_shape=IMG_SHAPE, include_top=False, weights="imagenet")
    base.trainable = False
    inp = keras.Input(shape=IMG_SHAPE)
    x = base(inp); x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x); x = layers.Dropout(0.4)(x)
    return keras.Model(inp, layers.Dense(NUM_CLASSES, activation="softmax")(x), name="efficientnetb0")

BUILDERS = {"custom_cnn": build_custom_cnn, "mobilenetv2": build_mobilenetv2, "efficientnetb0": build_efficientnetb0}

# --- Train and evaluate (with fine-tuning for transfer models) ---
def train_eval(name, train_ds, val_ds, test_ds, y_test, cw):
    model = BUILDERS[name]()
    lr = 1e-4 if name == "efficientnetb0" else 1e-3
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    cb = [EarlyStopping("val_loss", patience=PATIENCE, restore_best_weights=True, verbose=1),
          ReduceLROnPlateau("val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1)]
    model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, class_weight=cw, callbacks=cb, verbose=1)
    if name in ("mobilenetv2", "efficientnetb0"):
        base = model.layers[1]
        n, cut = len(base.layers), int((1 - UNFREEZE_FRACTION) * len(base.layers))
        for l in base.layers[:cut]: l.trainable = False
        for l in base.layers[cut:]: l.trainable = True
        model.compile(optimizer=keras.optimizers.Adam(FINETUNE_LR), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        print("Fine-tuning top 25% backbone...")
        model.fit(train_ds, validation_data=val_ds, epochs=FINETUNE_EPOCHS, class_weight=cw, callbacks=cb, verbose=1)
    y_pred = np.argmax(model.predict(test_ds), axis=1)
    acc = np.mean(y_test == y_pred)
    f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
    return {"model": model, "test_accuracy": acc, "test_f1_macro": f1,
            "classification_report": classification_report(y_test, y_pred, target_names=CLASS_NAMES, zero_division=0),
            "confusion_matrix": confusion_matrix(y_test, y_pred).tolist()}

# --- Train all, pick best, save ---
results = {}
for name in BUILDERS:
    print("\n" + "="*60 + "\nTraining", name.upper() + "\n" + "="*60)
    try:
        results[name] = train_eval(name, train_ds, val_ds, test_ds, y_test, class_weights)
        print("Test accuracy:", round(results[name]["test_accuracy"], 4), "| Test macro F1:", round(results[name]["test_f1_macro"], 4))
    except Exception as e:
        print("Error:", e)
        results[name] = None

best_name = max((n for n, r in results.items() if r), key=lambda n: results[n]["test_f1_macro"], default=None)
if best_name is None: raise RuntimeError("No model trained.")
best_f1 = results[best_name]["test_f1_macro"]
print("\nBest model:", best_name, "(test macro F1:", round(best_f1, 4), ")")
results[best_name]["model"].save(SAVED_MODEL_PATH)
with open(CLASS_NAMES_PATH, "w") as f: f.write("\n".join(CLASS_NAMES))
with open(os.path.join(MODELS_DIR, "best_model_metrics.json"), "w") as f:
    json.dump({"best_model": best_name, "test_accuracy": results[best_name]["test_accuracy"],
               "test_f1_macro": best_f1, "classification_report": results[best_name]["classification_report"],
               "confusion_matrix": results[best_name]["confusion_matrix"]}, f, indent=2)
print("Model saved →", SAVED_MODEL_PATH)
print("\nClassification report:\n", results[best_name]["classification_report"])

TensorFlow: 2.18.0 | Kaggle | Dataset: /kaggle/input/alzheimers-detection-dataset/dataset
Train: 6208 Val: 1280 Test: 640 | Class weights: {0: 1.2933333333333332, 1: 1.2933333333333332, 2: 0.6928571428571428, 3: 0.9897959183673469}

Training CUSTOM_CNN
Epoch 1/55
194/194 ━━━━━━━━━━━━━━━━━━━━ 68s 240ms/step - accuracy: 0.3039 - loss: 1.4936 - val_accuracy: 0.5250 - val_loss: 1.2263 - learning_rate: 0.0010
Epoch 2/55
194/194 ━━━━━━━━━━━━━━━━━━━━ 41s 212ms/step - accuracy: 0.3951 - loss: 1.3192 - val_accuracy: 0.5000 - val_loss: 3.5274 - learning_rate: 0.0010
Epoch 3/55
194/194 ━━━━━━━━━━━━━━━━━━━━ 41s 213ms/step - accuracy: 0.4126 - loss: 1.2677 - val_accuracy: 0.5547 - val_loss: 0.9782 - learning_rate: 0.0010
Epoch 4/55
194/194 ━━━━━━━━━━━━━━━━━━━━ 41s 213ms/step - accuracy: 0.4227 - loss: 1.2585 - val_accuracy: 0.2469 - val_loss: 1.5868 - learning_rate: 0.0010
Epoch 5/55
194/194 ━━━━━━━━━━━━━━━━━━━━ 41s 213ms/step - accuracy: 0.4313 - loss: 1.2366 - val_accuracy: 0.3828 - val_loss: 1.3

E0000 00:00:1771635500.265360      98 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1771635500.407644      98 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


194/194 ━━━━━━━━━━━━━━━━━━━━ 58s 89ms/step - accuracy: 0.2379 - loss: 2.0009 - val_accuracy: 0.0102 - val_loss: 1.8266 - learning_rate: 1.0000e-05
Epoch 2/15
194/194 ━━━━━━━━━━━━━━━━━━━━ 11s 55ms/step - accuracy: 0.2351 - loss: 1.8771 - val_accuracy: 0.1398 - val_loss: 1.3572 - learning_rate: 1.0000e-05
Epoch 3/15
194/194 ━━━━━━━━━━━━━━━━━━━━ 11s 56ms/step - accuracy: 0.2572 - loss: 1.7985 - val_accuracy: 0.1398 - val_loss: 1.9188 - learning_rate: 1.0000e-05
Epoch 4/15
194/194 ━━━━━━━━━━━━━━━━━━━━ 11s 56ms/step - accuracy: 0.2402 - loss: 1.8308 - val_accuracy: 0.1398 - val_loss: 2.1280 - learning_rate: 1.0000e-05
Epoch 5/15
193/194 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.2549 - loss: 1.8119
Epoch 5: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-06.
194/194 ━━━━━━━━━━━━━━━━━━━━ 11s 55ms/step - accuracy: 0.2549 - loss: 1.8119 - val_accuracy: 0.3500 - val_loss: 2.0475 - learning_rate: 1.0000e-05
Epoch 6/15
194/194 ━━━━━━━━━━━━━━━━━━━━ 11s 55ms/step - accuracy: 0.2